## 1) Phase 2: Silver Clean and Data Quality (Polars)

**Inputs:** `data/processed/bronze/dot_ontime/*.parquet` (excluding manifest parquet files)

**Outputs:**
- `data/processed/silver/fact_flights.parquet`
- `data/processed/silver/dim_airport.parquet`
- `data/processed/silver/dim_carrier.parquet`
- `reports/data_quality_phase2.md`

In [1]:
!pip -q install polars pyarrow pandas

import polars as pl
from pathlib import Path



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Paths
BRONZE_DIR = Path("../data/processed/bronze/dot_ontime")
SILVER_DIR = Path("../data/processed/silver")
REPORTS_DIR = Path("../reports")

SILVER_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Scan bronze parquet files (exclude any manifest parquet)
bronze_files = sorted([
    p for p in BRONZE_DIR.glob("*.parquet")
    if "manifest" not in p.name.lower()
])

if not bronze_files:
    raise FileNotFoundError(f"No parquet files found in {BRONZE_DIR}")

bronze_files[:5], len(bronze_files)


([WindowsPath('../data/processed/bronze/dot_ontime/flights_2013.parquet')], 1)

In [3]:
# Lazy scan bronze
lf = pl.scan_parquet([str(p) for p in bronze_files])
lf.collect_schema().names()[:40]


['FL_DATE',
 'AIRLINE',
 'AIRLINE_DOT',
 'AIRLINE_CODE',
 'DOT_CODE',
 'FL_NUMBER',
 'ORIGIN',
 'ORIGIN_CITY',
 'DEST',
 'DEST_CITY',
 'CRS_DEP_TIME',
 'DEP_TIME',
 'DEP_DELAY',
 'TAXI_OUT',
 'WHEELS_OFF',
 'WHEELS_ON',
 'TAXI_IN',
 'CRS_ARR_TIME',
 'ARR_TIME',
 'ARR_DELAY',
 'CANCELLED',
 'CANCELLATION_CODE',
 'DIVERTED',
 'CRS_ELAPSED_TIME',
 'ELAPSED_TIME',
 'AIR_TIME',
 'DISTANCE',
 'DELAY_DUE_CARRIER',
 'DELAY_DUE_WEATHER',
 'DELAY_DUE_NAS',
 'DELAY_DUE_SECURITY',
 'DELAY_DUE_LATE_AIRCRAFT']

In [4]:
# Normalize column names
def normalize_colname(c: str) -> str:
    c = c.strip().lower().replace(" ", "_").replace("-", "_")
    while "__" in c:
        c = c.replace("__", "_")
    return c

rename_map = {c: normalize_colname(c) for c in lf.collect_schema().names()}
lf = lf.rename(rename_map)

cols = set(lf.collect_schema().names())
sorted(list(cols))[:40]


['air_time',
 'airline',
 'airline_code',
 'airline_dot',
 'arr_delay',
 'arr_time',
 'cancellation_code',
 'cancelled',
 'crs_arr_time',
 'crs_dep_time',
 'crs_elapsed_time',
 'delay_due_carrier',
 'delay_due_late_aircraft',
 'delay_due_nas',
 'delay_due_security',
 'delay_due_weather',
 'dep_delay',
 'dep_time',
 'dest',
 'dest_city',
 'distance',
 'diverted',
 'dot_code',
 'elapsed_time',
 'fl_date',
 'fl_number',
 'origin',
 'origin_city',
 'taxi_in',
 'taxi_out',
 'wheels_off',
 'wheels_on']

In [5]:
# Helper: HHMM -> minutes since midnight
def hhmm_to_minutes(expr: pl.Expr) -> pl.Expr:
    s = expr.cast(pl.Utf8, strict=False).str.strip_chars()
    s = s.str.replace(r"\.0$", "", literal=False)  # remove trailing .0
    s4 = s.str.zfill(4)
    hh = s4.str.slice(0, 2).cast(pl.Int32, strict=False)
    mm = s4.str.slice(2, 2).cast(pl.Int32, strict=False)
    valid = (hh.is_between(0, 23)) & (mm.is_between(0, 59))
    return pl.when(valid).then(hh * 60 + mm).otherwise(None).cast(pl.Int32)


In [6]:
# ==============================
# COLUMN MAPPING (your schema)
# ==============================
# Based on your original columns:
# FL_DATE, AIRLINE, AIRLINE_CODE, FL_NUMBER, ORIGIN, DEST, CRS_DEP_TIME, DEP_TIME, ...

flight_date_col = "fl_date"

# Choose one:
carrier_col = "airline_code"   # recommended (DL, AA, UA)
# carrier_col = "airline"      # airline full name

flight_num_col = "fl_number"
origin_col = "origin"
dest_col = "dest"

crs_dep_col = "crs_dep_time"
dep_col = "dep_time"
crs_arr_col = "crs_arr_time"
arr_col = "arr_time"

dep_delay_col = "dep_delay"
arr_delay_col = "arr_delay"

cancelled_col = "cancelled"
diverted_col = "diverted"

taxi_out_col = "taxi_out"
taxi_in_col = "taxi_in"
air_time_col = "air_time"
distance_col = "distance"

# Delay causes (optional but great)
delay_carrier_col  = "delay_due_carrier"
delay_weather_col  = "delay_due_weather"
delay_nas_col      = "delay_due_nas"
delay_security_col = "delay_due_security"
delay_late_col     = "delay_due_late_aircraft"

# Quick sanity check: confirm columns exist
needed = [flight_date_col, carrier_col, flight_num_col, origin_col, dest_col, crs_dep_col, dep_col, crs_arr_col, arr_col]
{c: (c in cols) for c in needed}


{'fl_date': True,
 'airline_code': True,
 'fl_number': True,
 'origin': True,
 'dest': True,
 'crs_dep_time': True,
 'dep_time': True,
 'crs_arr_time': True,
 'arr_time': True}

In [7]:
# ==============================
# Build SILVER fact table
# ==============================

out = lf

# Parse date
out = out.with_columns(
    pl.col(flight_date_col).cast(pl.Utf8, strict=False).str.strip_chars().alias("_flight_date_str")
).with_columns(
    pl.col("_flight_date_str").str.strptime(pl.Date, strict=False).alias("flight_date")
).drop("_flight_date_str")

# Times -> minutes
out = out.with_columns(
    hhmm_to_minutes(pl.col(crs_dep_col)).alias("crs_dep_min") if crs_dep_col else pl.lit(None, dtype=pl.Int32).alias("crs_dep_min"),
    hhmm_to_minutes(pl.col(dep_col)).alias("dep_min") if dep_col else pl.lit(None, dtype=pl.Int32).alias("dep_min"),
    hhmm_to_minutes(pl.col(crs_arr_col)).alias("crs_arr_min") if crs_arr_col else pl.lit(None, dtype=pl.Int32).alias("crs_arr_min"),
    hhmm_to_minutes(pl.col(arr_col)).alias("arr_min") if arr_col else pl.lit(None, dtype=pl.Int32).alias("arr_min"),
)

# Delays
out = out.with_columns(
    pl.col(dep_delay_col).cast(pl.Float32, strict=False).alias("dep_delay_min") if dep_delay_col else pl.lit(None, dtype=pl.Float32).alias("dep_delay_min"),
    pl.col(arr_delay_col).cast(pl.Float32, strict=False).alias("arr_delay_min") if arr_delay_col else pl.lit(None, dtype=pl.Float32).alias("arr_delay_min"),
)

# Flags
out = out.with_columns(
    pl.col(cancelled_col).cast(pl.Int8, strict=False).fill_null(0).alias("is_cancelled") if cancelled_col else pl.lit(0, dtype=pl.Int8).alias("is_cancelled"),
    pl.col(diverted_col).cast(pl.Int8, strict=False).fill_null(0).alias("is_diverted") if diverted_col else pl.lit(0, dtype=pl.Int8).alias("is_diverted"),
)

# Canonical IDs
out = out.with_columns(
    pl.col(carrier_col).cast(pl.Utf8, strict=False).str.strip_chars().alias("carrier") if carrier_col else pl.lit(None, dtype=pl.Utf8).alias("carrier"),
    pl.col(flight_num_col).cast(pl.Int32, strict=False).alias("flight_num") if flight_num_col else pl.lit(None, dtype=pl.Int32).alias("flight_num"),
    pl.col(origin_col).cast(pl.Utf8, strict=False).str.strip_chars().alias("origin") if origin_col else pl.lit(None, dtype=pl.Utf8).alias("origin"),
    pl.col(dest_col).cast(pl.Utf8, strict=False).str.strip_chars().alias("dest") if dest_col else pl.lit(None, dtype=pl.Utf8).alias("dest"),
)

# Time features
out = out.with_columns(
    pl.col("flight_date").dt.year().alias("year"),
    pl.col("flight_date").dt.month().alias("month"),
    pl.col("flight_date").dt.weekday().alias("dow"),
    (pl.col("crs_dep_min") / 60).floor().cast(pl.Int8).alias("dep_hour"),
)

# Delay labels
out = out.with_columns(
    (pl.col("dep_delay_min") >= 15).cast(pl.Int8).fill_null(0).alias("is_dep_delayed_15"),
    (pl.col("arr_delay_min") >= 15).cast(pl.Int8).fill_null(0).alias("is_arr_delayed_15"),
    (pl.col("arr_delay_min") >= 60).cast(pl.Int8).fill_null(0).alias("is_severe_arr_delay_60"),
)

# Optional numeric fields
def opt_num(colname, alias, dtype):
    if colname and colname in cols:
        return pl.col(colname).cast(dtype, strict=False).alias(alias)
    return pl.lit(None, dtype=dtype).alias(alias)

out = out.with_columns(
    opt_num(taxi_out_col, "taxi_out_min", pl.Float32),
    opt_num(taxi_in_col, "taxi_in_min", pl.Float32),
    opt_num(air_time_col, "air_time_min", pl.Float32),
    opt_num(distance_col, "distance_miles", pl.Float32),
)

# Optional delay-cause fields
def opt_delay(colname, alias):
    if colname and colname in cols:
        return pl.col(colname).cast(pl.Float32, strict=False).alias(alias)
    return pl.lit(None, dtype=pl.Float32).alias(alias)

out = out.with_columns(
    opt_delay(delay_carrier_col, "delay_due_carrier"),
    opt_delay(delay_weather_col, "delay_due_weather"),
    opt_delay(delay_nas_col, "delay_due_nas"),
    opt_delay(delay_security_col, "delay_due_security"),
    opt_delay(delay_late_col, "delay_due_late_aircraft"),
)

# Keep columns
keep = [
    "flight_date", "year", "month", "dow",
    "carrier", "flight_num", "origin", "dest",
    "crs_dep_min", "dep_min", "crs_arr_min", "arr_min",
    "dep_delay_min", "arr_delay_min",
    "is_cancelled", "is_diverted",
    "is_dep_delayed_15", "is_arr_delayed_15", "is_severe_arr_delay_60",
    "dep_hour",
    "taxi_out_min", "taxi_in_min", "air_time_min", "distance_miles",
    "delay_due_carrier", "delay_due_weather", "delay_due_nas", "delay_due_security", "delay_due_late_aircraft",
]
out = out.select([c for c in keep if c in out.collect_schema().names()])

# Drop broken rows
out = out.filter(
    pl.col("flight_date").is_not_null() &
    pl.col("origin").is_not_null() &
    pl.col("dest").is_not_null()
)

# Materialize
fact = out.collect()
fact.shape, fact.head(3)


((3000000, 29),
 shape: (3, 29)
 ┌─────────────┬──────┬───────┬─────┬───┬──────────────┬──────────────┬──────────────┬──────────────┐
 │ flight_date ┆ year ┆ month ┆ dow ┆ … ┆ delay_due_we ┆ delay_due_na ┆ delay_due_se ┆ delay_due_la │
 │ ---         ┆ ---  ┆ ---   ┆ --- ┆   ┆ ather        ┆ s            ┆ curity       ┆ te_aircraft  │
 │ date        ┆ i32  ┆ i8    ┆ i8  ┆   ┆ ---          ┆ ---          ┆ ---          ┆ ---          │
 │             ┆      ┆       ┆     ┆   ┆ f32          ┆ f32          ┆ f32          ┆ f32          │
 ╞═════════════╪══════╪═══════╪═════╪═══╪══════════════╪══════════════╪══════════════╪══════════════╡
 │ 2019-01-09  ┆ 2019 ┆ 1     ┆ 3   ┆ … ┆ null         ┆ null         ┆ null         ┆ null         │
 │ 2022-11-19  ┆ 2022 ┆ 11    ┆ 6   ┆ … ┆ null         ┆ null         ┆ null         ┆ null         │
 │ 2022-07-22  ┆ 2022 ┆ 7     ┆ 5   ┆ … ┆ null         ┆ null         ┆ null         ┆ null         │
 └─────────────┴──────┴───────┴─────┴───┴─────────

In [8]:
# Save Silver outputs
fact_path = SILVER_DIR / "fact_flights.parquet"
fact.write_parquet(fact_path)

# Dimensions
dim_airport = (
    pl.concat([
        fact.select(pl.col("origin").alias("airport")),
        fact.select(pl.col("dest").alias("airport")),
    ], how="vertical")
    .unique()
    .sort("airport")
)

dim_carrier = (
    fact.select(pl.col("carrier"))
        .unique()
        .sort("carrier")
)

dim_airport.write_parquet(SILVER_DIR / "dim_airport.parquet")
dim_carrier.write_parquet(SILVER_DIR / "dim_carrier.parquet")

fact_path


WindowsPath('../data/processed/silver/fact_flights.parquet')

In [9]:
# =========================================================
# PHASE 2 — DATA QUALITY REPORT
# Output: reports/data_quality_phase2.md
# Requires: fact (Polars DataFrame)
# =========================================================

n = fact.height
if n == 0:
    raise ValueError("fact table is empty. Check filters or parsing.")

null_rates = (
    fact.null_count()
        .transpose(include_header=True, header_name="col", column_names=["nulls"])
        .with_columns((pl.col("nulls") / n).alias("null_rate"))
        .sort("null_rate", descending=True)
)

unique_airports = (
    pl.concat(
        [
            fact.select(pl.col("origin").alias("airport")),
            fact.select(pl.col("dest").alias("airport")),
        ],
        how="vertical",
    )
    .select(pl.col("airport").n_unique())
    .item()
)

kpis = {
    "rows": n,
    "min_date": fact.select(pl.min("flight_date")).item(),
    "max_date": fact.select(pl.max("flight_date")).item(),
    "unique_carriers": fact.select(pl.col("carrier").n_unique()).item() if "carrier" in fact.columns else None,
    "unique_airports": unique_airports,
    "cancel_rate": fact.select(pl.mean("is_cancelled")).item(),
    "divert_rate": fact.select(pl.mean("is_diverted")).item(),
    "arr_delay_15_rate": fact.select(pl.mean("is_arr_delayed_15")).item(),
    "severe_arr_delay_60_rate": fact.select(pl.mean("is_severe_arr_delay_60")).item(),
}

delay_stats = fact.select([
    pl.col("arr_delay_min").min().alias("arr_delay_min_min"),
    pl.col("arr_delay_min").quantile(0.5).alias("arr_delay_min_median"),
    pl.col("arr_delay_min").max().alias("arr_delay_min_max"),
    pl.col("dep_delay_min").min().alias("dep_delay_min_min"),
    pl.col("dep_delay_min").quantile(0.5).alias("dep_delay_min_median"),
    pl.col("dep_delay_min").max().alias("dep_delay_min_max"),
])

top_airports = (
    pl.concat(
        [
            fact.select(pl.col("origin").alias("airport")),
            fact.select(pl.col("dest").alias("airport")),
        ],
        how="vertical",
    )
    .group_by("airport")
    .agg(pl.len().alias("flights"))
    .sort("flights", descending=True)
    .head(15)
)

cause_cols = [
    "delay_due_carrier",
    "delay_due_weather",
    "delay_due_nas",
    "delay_due_security",
    "delay_due_late_aircraft",
]
existing_cause_cols = [c for c in cause_cols if c in fact.columns]

delay_causes_summary = None
if existing_cause_cols:
    delay_causes_summary = fact.select([
        pl.mean(c).alias(f"avg_{c}") for c in existing_cause_cols
    ])

out_md = REPORTS_DIR / "data_quality_phase2.md"
with out_md.open("w", encoding="utf-8") as f:
    f.write("# Phase 2 Data Quality Report\n\n")

    f.write("## Summary KPIs\n")
    for k, v in kpis.items():
        f.write(f"- **{k}**: {v}\n")

    f.write("\n## Delay Minutes Sanity Stats\n\n")
    f.write(delay_stats.to_pandas().to_markdown(index=False))
    f.write("\n")

    if delay_causes_summary is not None:
        f.write("\n## Average Delay Minutes by Cause\n\n")
        f.write(delay_causes_summary.to_pandas().to_markdown(index=False))
        f.write("\n")

    f.write("\n## Null Rates (Top 25)\n\n")
    f.write(null_rates.head(25).to_pandas().to_markdown(index=False))
    f.write("\n")

    f.write("\n## Top Airports by Total Flights (Origin + Dest)\n\n")
    f.write(top_airports.to_pandas().to_markdown(index=False))
    f.write("\n")

print(f"✅ Data quality report written → {out_md.resolve()}")


✅ Data quality report written → C:\Users\Ti Em\Desktop\Practice Projects\Data Analysis\airline-intelligence-platform\reports\data_quality_phase2.md
